In [2]:
import argparse
import logging
import time
from pathlib import Path
from typing import List, Optional
from datetime import datetime

import pandas as pd
import yfinance as yf

# Set up basic logging (standard in quant pipelines)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger("YFDataPipeline")

# ==========================================
# Step 1: Parse Args
# ==========================================
def parse_args(args_list: Optional[List[str]] = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Fetch Top 5000 US companies historical data from Yahoo Finance.")
    parser.add_argument("--input_file", type=str, default="top_5000_us_by_marketcap.csv", help="Path to the dataset containing tickers.")
    parser.add_argument("--output_file", type=str, default="top_5000_yf_data.pkl", help="Output path (Pickle helps perfectly retain Multi-Index columns without flattening).")
    parser.add_argument("--start_date", type=str, default="2010-01-01", help="Data start date (YYYY-MM-DD).")
    parser.add_argument("--end_date", type=str, default=datetime.today().strftime('%Y-%m-%d'), help="Data end date (YYYY-MM-DD).")
    parser.add_argument("--batch_size", type=int, default=500, help="Number of symbols to fetch concurrently to respect YF rate limits.")
    parser.add_argument("--max_retries", type=int, default=3, help="Max retry attempts for failed ticker downloads.")
    
    # In Jupyter, sys.argv behavior is different. Passing empty list defaults to the argument defaults above.
    return parser.parse_args(args_list if args_list is not None else [])


# ==========================================
# Step 2: Call API (With Retry Logic)
# ==========================================
def fetch_yf_data(tickers: List[str], start: str, end: str, batch_size: int, max_retries: int) -> pd.DataFrame:
    logger.info(f"Fetching API data for {len(tickers)} tickers from {start} to {end}...")
    
    all_dfs = []
    pending_tickers = tickers.copy()
    
    for attempt in range(max_retries):
        if not pending_tickers:
            break
            
        if attempt > 0:
            logger.warning(f"--- Retry Attempt {attempt}/{max_retries - 1} for {len(pending_tickers)} failed tickers ---")
            time.sleep(2) # Give the YF server a short breather before retrying
            
        attempt_dfs = []
        
        # Process in batches
        for i in range(0, len(pending_tickers), batch_size):
            batch = pending_tickers[i : i + batch_size]
            logger.info(f"Downloading batch {i//batch_size + 1}/{(len(pending_tickers)-1)//batch_size + 1} ({len(batch)} tickers)")
            
            # yfinance creates a MultiIndex: Columns -> (Feature, Ticker) i.e. (Close, AAPL)
            df_batch = yf.download(
                tickers=batch,
                start=start,
                end=end,
                group_by='column',
                threads=True, 
                progress=False,
                auto_adjust=False, # <--- Request RAW unadjusted prices
                # actions=True       # <--- Request Dividends and Stock Splits
            )
            
            # If batch has 1 ticker, yfinance sometimes flattens it. We force MultiIndex behavior.
            if not df_batch.empty and not isinstance(df_batch.columns, pd.MultiIndex) and len(batch) == 1:
                df_batch.columns = pd.MultiIndex.from_product([df_batch.columns, batch])
            
            if not df_batch.empty:
                attempt_dfs.append(df_batch)
            time.sleep(1) # Soft rate-limit mitigation
            
        if not attempt_dfs:
            continue
            
        df_attempt_combined = pd.concat(attempt_dfs, axis=1)
        
        # Identify successfully retrieved tickers (those having at least one non-NaN 'Close' value)
        if 'Close' in df_attempt_combined.columns.levels[0]:
            close_df = df_attempt_combined.xs('Close', level=0, axis=1)
            valid_mask = ~close_df.isna().all()
            successful_this_round = valid_mask[valid_mask].index.tolist()
            
            if successful_this_round:
                idx = pd.IndexSlice
                df_success = df_attempt_combined.loc[:, idx[:, successful_this_round]]
                all_dfs.append(df_success)
            
            # Anything not successfully pulled gets added to the pending queue for the next loop attempt
            pending_tickers = [t for t in pending_tickers if t not in successful_this_round]
        else:
            logger.error("No 'Close' columns found in this attempt!")
            pending_tickers = [] # Break out safely

    logger.info(f"API Calls complete. Successfully downloaded {len(tickers) - len(pending_tickers)}/{len(tickers)} tickers.")
    if pending_tickers:
        logger.error(f"Ultimately failed to download {len(pending_tickers)} tickers despite {max_retries} attempts. Continuing without them...")
        
    if not all_dfs:
        raise ValueError("All API calls failed. No data fetched.")
        
    # Combine all successfull attempts
    df_final = pd.concat(all_dfs, axis=1)
    
    # Clean up column index sorting so that (Adj Close, Close, High, Low...) are grouped nicely
    df_final = df_final.sort_index(axis=1)
    
    return df_final


# ==========================================
# Step 3: Filter Data
# ==========================================
def filter_and_clean_data(df: pd.DataFrame, min_active_days: int = 50) -> pd.DataFrame:
    logger.info("Filtering and checking data quality...")
    original_shape = df.shape
    
    # 1. Drop days (rows) where the entire market universe has zero data (e.g., erratic holidays fetched)
    df = df.dropna(axis=0, how='all')
    
    # # 2. Filter out bad tickers, but allow Recent IPOs to survive
    # if 'Close' in df.columns.levels[0]:
    #     close_data = df.xs('Close', level=0, axis=1) # Extract just 'Close' section
        
    #     # We previously dropped tickers with > 50% missing data.
    #     # This aggressively murdered recent IPOs (like a 2024 IPO being 80% NaNs because it didn't exist in 2020-2023).
    #     # Fix: Now we just require the stock to have at least `min_active_days` of valid historical trading data.
    #     valid_tickers = close_data.columns[close_data.notna().sum() >= min_active_days]
        
    #     # Keep only these valid tickers across all feature levels
    #     idx = pd.IndexSlice
    #     df = df.loc[:, idx[:, valid_tickers]]
        
    # 3. Handle Missing Values Carefully
    # We must NOT forward-fill corporate actions (Dividends/Splits), they should strictly default to 0.0
    idx = pd.IndexSlice
    features = df.columns.levels[0]

    price_vol_features = [f for f in features if f not in ['Dividends', 'Stock Splits']]
    action_features = [f for f in features if f in ['Dividends', 'Stock Splits']]
    
    # Forward-fill gaps for Price and Volume data (limit 5 days for typical maximum trading halts)
    if price_vol_features:
        df.loc[:, idx[price_vol_features, :]] = df.loc[:, idx[price_vol_features, :]].ffill(limit=5).bfill(limit=1)
        
    # Zero-fill gaps for Corporate Actions
    if action_features:
        df.loc[:, idx[action_features, :]] = df.loc[:, idx[action_features, :]].fillna(0.0)
    
    logger.info(f"Data filtering complete. Final shape: {df.shape} (Removed {original_shape[1] - df.shape[1]} columns)")
    return df


# ==========================================
# Step 4: Store Data
# ==========================================
def store_data(df: pd.DataFrame, output_path: str) -> None:
    logger.info(f"Storing multi-indexed matrix to {output_path}...")
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    
    # We use pickle here because it perfectly preserves pandas MultiIndex (Parquet / CSV often flattens them)
    df.to_pickle(output_path)
    logger.info("Storage phase complete.")


# ==========================================
# Execution Pipeline
# ==========================================
def main(args_list: Optional[List[str]] = None) -> pd.DataFrame:
    # 1. Parse Arguments
    args = parse_args(args_list)
    
    # Load Universe
    universe_df = pd.read_csv(args.input_file)
    
    # Tidy up tickers (Yahoo uses hyphens instead of dots/slashes for structures like BRK/A -> BRK-A)
    # We replace '.' and '/' with '-', and strip whitespace
    universe_tickers = universe_df['symbol'].astype(str).str.strip().str.replace('.', '-', regex=False).str.replace('/', '-', regex=False).dropna().unique().tolist()
    
    # 2. Call API
    raw_df = fetch_yf_data(universe_tickers, args.start_date, args.end_date, args.batch_size, args.max_retries)
    
    # 3. Filter Data
    clean_df = filter_and_clean_data(raw_df)
    
    # 4. Store Data
    store_data(clean_df, args.output_file)
    
    return clean_df

# To run this directly in the Jupyter Notebook:
df_historical = main([]) 
df_historical.head()

2026-04-13 11:50:49,318 - YFDataPipeline - INFO - Fetching API data for 4999 tickers from 2010-01-01 to 2026-04-13...
2026-04-13 11:50:49,319 - YFDataPipeline - INFO - Downloading batch 1/10 (500 tickers)
2026-04-13 11:52:28,327 - YFDataPipeline - INFO - Downloading batch 2/10 (500 tickers)
2026-04-13 11:55:17,036 - yfinance - ERROR - 
13 Failed downloads:
2026-04-13 11:55:17,058 - yfinance - ERROR - ['APA', 'ONON', 'ABVX', 'HPQ', 'HUM', 'EDU', 'CLX', 'ALGN', 'RMBS', 'RS', 'BNT', 'CX', 'RGC']: TypeError("'NoneType' object is not subscriptable")
2026-04-13 11:55:29,585 - YFDataPipeline - INFO - Downloading batch 3/10 (500 tickers)
2026-04-13 11:57:20,290 - YFDataPipeline - INFO - Downloading batch 4/10 (500 tickers)
2026-04-13 11:58:59,574 - YFDataPipeline - INFO - Downloading batch 5/10 (500 tickers)
2026-04-13 12:00:28,216 - YFDataPipeline - INFO - Downloading batch 6/10 (500 tickers)
2026-04-13 12:02:05,723 - YFDataPipeline - INFO - Downloading batch 7/10 (500 tickers)
2026-04-13 12:

Price       Adj Close                                                     \
Ticker              A         AA      AACG       AAL      AAME AAMI AAOI   
Date                                                                       
2010-01-04  19.810984  35.301849  0.377391  4.496877  1.164885  NaN  NaN   
2010-01-05  19.595791  34.199337  0.373376  5.005958  1.235484  NaN  NaN   
2010-01-06  19.526165  35.980320  0.362135  4.798553  1.235484  NaN  NaN   
2010-01-07  19.500841  35.217033  0.362938  4.939965  1.164885  NaN  NaN   
2010-01-08  19.494518  36.086338  0.381406  4.845690  1.226659  NaN  NaN   

Price                                 ... Volume                             \
Ticker          AAON        AAP AAPG  ...   ZTEK ZTO ZTS     ZUMZ ZURA ZVIA   
Date                                  ...                                     
2010-01-04  3.463518  34.588707  NaN  ...    NaN NaN NaN   268400  NaN  NaN   
2010-01-05  3.363025  34.383133  NaN  ...    NaN NaN NaN   812300  NaN  NaN   
2010-01-06  3.229613  34.682930  NaN  ...    NaN NaN NaN   507200  NaN  NaN   
2010-01-07  3.349164  34.674366  NaN  ...    NaN NaN NaN  1857900  NaN  NaN   
2010-01-08  3.389015  34.811405  NaN  ...    NaN NaN NaN   334300  NaN  NaN   

Price                          
Ticker     ZVRA ZWS ZYBT ZYME  
Date                           
2010-01-04  NaN NaN  NaN  NaN  
2010-01-05  NaN NaN  NaN  NaN  
2010-01-06  NaN NaN  NaN  NaN  
2010-01-07  NaN NaN  NaN  NaN  
2010-01-08  NaN NaN  NaN  NaN  

[5 rows x 29994 columns]

In [1]:
aapl_data= df_historical.xs('AAPL', level=1, axis=1)
print(aapl_data)

NameError: name 'df_historical' is not defined